# 01 — Data Pipeline

## Task 1.2 — Open-Meteo weather fetch

Fetches hourly `temperature_2m` and `shortwave_radiation` (GHI) from the
Open-Meteo historical archive for a given location and date range.

In [ ]:
import ssl
import requests
import pandas as pd
from requests.adapters import HTTPAdapter

In [ ]:
class _WinCertAdapter(HTTPAdapter):
    """Mounts Windows system CA store so requests works behind corporate proxies."""
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.load_default_certs(ssl.Purpose.SERVER_AUTH)
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount("https://", _WinCertAdapter())


def fetch_weather(lat, lon, start_date, end_date, timezone) -> pd.DataFrame:
    """Returns hourly tz-aware DataFrame indexed by timestamp,
       with columns: temperature_2m, shortwave_radiation (this is your GHI/irradiance)."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,shortwave_radiation",
        "timezone": timezone,
    }
    resp = _session.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()

    hourly = resp.json()["hourly"]
    df = pd.DataFrame({
        "temperature_2m":      hourly["temperature_2m"],
        "shortwave_radiation": hourly["shortwave_radiation"],
    }, index=pd.to_datetime(hourly["time"]))
    df.index = df.index.tz_localize(timezone)
    df.index.name = "timestamp"
    return df

In [ ]:
# Quick smoke-test — Austin, TX, one week
df_weather = fetch_weather(
    lat=30.27,
    lon=-97.74,
    start_date="2018-01-01",
    end_date="2018-01-07",
    timezone="America/Chicago",
)

print(df_weather.head())
print("\nIndex dtype:", df_weather.index.dtype)
print("Shape:", df_weather.shape)